# HAM10000 class-conditional DDPM


# Phase 1: Setup


## 1.1 Check runtime


In [ ]:
!nvidia-smi

## 1.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.3 Project and data paths


In [ ]:
import os
from pathlib import Path

# Project directory on the mounted Drive.
PROJECT_DIR = '/content/drive/MyDrive/ddpm-derm-augmentation'
DATA_DIR    = PROJECT_DIR + '/data'
OUTPUTS_DIR = PROJECT_DIR + '/outputs'
LOCAL_CKPT_DIR = '/content/ddpm_ckpt'
SNAPSHOT_DIR = OUTPUTS_DIR + '/ddpm/checkpoints'

os.environ['DDPM_DERM_DATA_DIR'] = DATA_DIR
os.environ['DDPM_DERM_OUTPUTS_DIR'] = OUTPUTS_DIR
os.environ['DDPM_DERM_LOCAL_CKPT_DIR'] = LOCAL_CKPT_DIR

assert Path(PROJECT_DIR, 'src', 'ddpm_derm', 'config.py').is_file(), \
    f'PROJECT_DIR wrong: no src/ddpm_derm/config.py under {PROJECT_DIR}'
assert Path(DATA_DIR, 'manifests', 'class_to_idx.json').is_file(), \
    f'DATA_DIR wrong: no manifests/class_to_idx.json under {DATA_DIR}'
assert Path(SNAPSHOT_DIR).is_dir(), f'SNAPSHOT_DIR missing: {SNAPSHOT_DIR}'
print('paths OK')
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR    =', DATA_DIR)
print('OUTPUTS_DIR =', OUTPUTS_DIR)
print('LOCAL_CKPT_DIR =', LOCAL_CKPT_DIR)
print('SNAPSHOT_DIR =', SNAPSHOT_DIR)

## 1.4 Copy data to local disk


In [ ]:
import subprocess
import pandas as pd

DATA_DIR = '/content/data'

def _manifest_status():
    try:
        rows = pd.concat([
            pd.read_csv(Path(DATA_DIR, 'manifests', f'{split}.csv'))
            for split in ('train', 'val', 'test')
        ], ignore_index=True)
    except FileNotFoundError:
        return None, None
    missing = [p for p in rows['image_path'] if not Path(DATA_DIR, p).is_file()]
    return rows, missing

_rows, _missing = _manifest_status()
_ready = (_rows is not None and len(_rows) == 10015
          and _rows['image_path'].nunique() == 10015 and not _missing)
if _ready:
    print('local data already complete; skipping Drive copy')
else:
    print('copying data from Drive to /content/data (may be silent for a while) ...')
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-a', PROJECT_DIR + '/data/.', DATA_DIR + '/'], check=True)
    _rows, _missing = _manifest_status()

os.environ['DDPM_DERM_DATA_DIR'] = DATA_DIR
assert _rows is not None, 'local manifests are still missing after copy'
assert len(_rows) == 10015, f'unexpected manifest rows: {len(_rows)}'
assert _rows['image_path'].nunique() == 10015, 'manifest paths are not unique'
assert not _missing, f'missing manifest images: {len(_missing)}'
print('using local data:', DATA_DIR)
print('manifest rows: 10015; missing: 0')

## 1.5 Install dependencies


In [ ]:
!pip install -q diffusers

# Phase 2: Smoke test


## 2.1 DDPM smoke test

In [ ]:
!cd "{PROJECT_DIR}" && python scripts/smoke_ddpm.py

# Phase 3: Real run


## 3.1 Train the DDPM to epoch 100


In [ ]:
# Safe default: resume only. Remove --resume only for an intentional fresh run.
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.train_ddpm --epochs 100 --img-size 64 --batch-size 64 --ema-decay 0.999 --preview-every 5 --resume --output-dir "{LOCAL_CKPT_DIR}" --snapshot-dir "{SNAPSHOT_DIR}" --snapshot-every 10

## 3.2 Epoch-100 sampling, validation, and publication

Enabling the commented cleanup deletes local staging.


In [ ]:
# [manual] 3.2a Restore the epoch-100 snapshot to local disk, then sample 500 df to staging.
import shutil
from pathlib import Path

EPOCH      = 100
SNAP       = Path(SNAPSHOT_DIR) / f'run_seed0_epoch{EPOCH:04d}.pt'
LOCAL_CKPT = Path(LOCAL_CKPT_DIR) / SNAP.name
STAGING    = '/content/synthetic_df_epoch0100_seed0'

assert SNAP.is_file(), (
    f'epoch-{EPOCH} snapshot not on Drive yet: {SNAP}\n'
    f'train to epoch {EPOCH} first (3.1); snapshots are written every 10 epochs')
Path(LOCAL_CKPT_DIR).mkdir(parents=True, exist_ok=True)
if not (LOCAL_CKPT.is_file() and LOCAL_CKPT.stat().st_size == SNAP.stat().st_size):
    print(f'copying snapshot to local disk: {SNAP} -> {LOCAL_CKPT}')
    shutil.copy2(SNAP, LOCAL_CKPT)   # copy only; the Drive snapshot stays immutable
print('local checkpoint ready:', LOCAL_CKPT)

# !rm -rf "{STAGING}"   # uncomment ONLY to clear a failed partial staging run

# Require epoch 100, EMA weights, and an empty output directory.
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.sample_ddpm --ckpt "{LOCAL_CKPT}" --require-epoch 100 --require-ema --n 500 --num-steps 50 --eta 0.0 --seed 0 --out-dir "{STAGING}"

In [ ]:
# [manual] 3.2b Validate staging -> copy whole folder to a NEW versioned Drive
# dir -> re-validate on Drive -> only then write _READY.json. Refuses an
# existing destination; a failed attempt leaves no _READY marker.
FINAL = OUTPUTS_DIR + '/synthetic_df/epoch0100_seed0'
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.publish_synthetic --src "{STAGING}" --dest "{FINAL}" --expect-n 500 --expect-epoch 100 --expect-seed 0 --expect-steps 50

In [ ]:
# [manual] 3.2c Final state of the published set (torch-free; safe to re-run).
import json
import pandas as pd
from pathlib import Path

final = Path(OUTPUTS_DIR) / 'synthetic_df' / 'epoch0100_seed0'
manifest = final / 'synthetic_df.csv'
assert manifest.is_file(), f'not published yet: no {manifest}'
rows = pd.read_csv(manifest)
missing = [p for p in rows['image_path'] if not (final / p).is_file()]
print('final dir     :', final)
print('manifest rows :', len(rows))
print('missing files :', len(missing))
print('metadata      :', json.dumps(json.loads((final / 'metadata.json').read_text()), indent=2))
ready = final / '_READY.json'
if ready.is_file():
    print('_READY.json   : PRESENT ->', json.loads(ready.read_text())['published_utc'])
else:
    print('_READY.json   : MISSING -> the set is NOT complete; do not train C4 on it')

## 3.3 Show previews and NN montage


In [ ]:
from pathlib import Path
from IPython.display import Image, display

SAMPLES = Path(OUTPUTS_DIR) / 'ddpm' / 'samples'
SYN     = Path(OUTPUTS_DIR) / 'synthetic_df'

previews = sorted(SAMPLES.glob('preview_*.png'))
if previews:
    print('latest df preview grid:', previews[-1].name)
    display(Image(str(previews[-1])))
else:
    print('no preview grids yet -- run 3.1 with --preview-every > 0')

montages = sorted(SYN.glob('*/nn_check.png'))
if montages:
    for m in montages:
        print(f'NN check [{m.parent.name}] (left = generated, right = nearest real train df):')
        display(Image(str(m)))
else:
    print('no versioned NN montages yet -- run 3.2 first')
    print('(the legacy epoch-60 montage lives at outputs/figures/ddpm_nn_check_df.png)')